# EDA · DataSetTechmind V5 (perfilado)

**Entrega DS:** solo este notebook + `data/DataSetTechmindV5_perfilado.csv`.  
**Carga:** ruta relativa al paquete (no hardcode de máquina, no ETL).  
**No incluye:** ETL, train LogReg, joblib, re-perfilado.

## Fuentes de práctica (EDA de calidad)

| Referencia | Uso en este notebook |
|------------|----------------------|
| **Tukey — EDA** | Describir, visualizar, generar hipótesis; no modelar |
| **IBM — EDA** | Calidad, univariado/multivariado, patrones y anomalías *antes* del modelo |
| **CRISP-DM Data Understanding** | Schema, distribuciones, relaciones, documentación de hallazgos |
| **pandas / seaborn** | Tablas + gráficos reproducibles |

## Stack permitido

`pandas` · `numpy` · `matplotlib` · `seaborn` (+ stdlib). **Sin** `sklearn` en este notebook.

## Salidas

- Figuras embebidas en el notebook **y** PNG en `docs/figures/` (si hay carpeta escribible).
- `docs/EDA_HITO2_CALIDAD_DECISIONES.json` + `docs/EDA_DataSetTechmindV5_HALLAZGOS.md`


## 0. Setup (backend de gráficos + estilo)


In [ ]:
# Backend inline: sin esto, en muchos entornos Windows/VS Code
# matplotlib usa Agg y `plt.show()` NO inserta imagen en el notebook.
%matplotlib inline

from __future__ import annotations

import json
import re
import warnings
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

# Estilo reproducible (seaborn sobre matplotlib)
sns.set_theme(style="whitegrid", context="notebook", font_scale=0.95)
plt.rcParams.update({
    "figure.figsize": (10, 4.5),
    "figure.dpi": 110,
    "savefig.dpi": 140,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
})

# Contrato del perfilado A0 (embebido: entrega = CSV + este notebook)
EXPECTED_ROWS = 1061
L1_ORDER = [
    "Arquitectura",
    "Bases_de_Datos",
    "Hardware",
    "Inteligencia_Artificial",
    "Lenguajes_Programacion",
    "Redes_y_Comunicaciones",
    "Sistemas_Operativos",
]
SCHEMA = [
    "id_fragmento",
    "titulo_origen",
    "categoria_l1",
    "pagina",
    "texto_crudo",
    "longitud_caracteres",
    "fecha_extraccion",
    "hash_texto",
    "texto_limpio",
    "conteo_tokens",
    "densidad_lexica",
]
# TOKEN_PATTERN A0 (conectores técnicos); solo para métricas de vocabulario en EDA
TOKEN_PATTERN = r"[A-Za-zÁÉÍÓÚáéíóúñÑ0-9_]+(?:[.\-/][A-Za-zÁÉÍÓÚáéíóúñÑ0-9_]+)*"
TOKEN_RE = re.compile(TOKEN_PATTERN)

print("matplotlib backend =", plt.get_backend())
print("seaborn =", sns.__version__, "| pandas =", pd.__version__)


## 1. Cargar el CSV perfilado entregado

El dataset va **en el mismo paquete** que este notebook:

```text
paquete/
├── data/DataSetTechmindV5_perfilado.csv   ← único input
├── notebooks/02_EDA_….ipynb               ← este archivo
└── docs/figures/                          ← salida de gráficos
```

Ruta **relativa** (sin `C:\…`, sin monorepo, sin ETL).


In [ ]:
# CSV de la entrega: siempre data/DataSetTechmindV5_perfilado.csv
# Cwd recomendado = raíz del paquete. Si Jupyter arranca en notebooks/, se usa ../data/
_CSV_NAME = "DataSetTechmindV5_perfilado.csv"
_candidates = [
    Path("data") / _CSV_NAME,           # cwd = raíz del paquete
    Path("..") / "data" / _CSV_NAME,    # cwd = notebooks/
    Path(_CSV_NAME),                    # CSV junto al notebook (opcional)
]
PATH_PERFILADO = next((p for p in _candidates if p.is_file()), None)
if PATH_PERFILADO is None:
    raise FileNotFoundError(
        "No está el CSV perfilado de la entrega.\n"
        "Esperado: data/DataSetTechmindV5_perfilado.csv\n"
        f"cwd actual: {Path.cwd()}\n"
        "Abra Jupyter en la raíz del paquete (o en notebooks/)."
    )

# Salidas del EDA junto al data/ del paquete
PACKAGE_ROOT = PATH_PERFILADO.parent.parent if PATH_PERFILADO.parent.name == "data" else Path.cwd()
PATH_DOCS = PACKAGE_ROOT / "docs"
PATH_FIGS = PATH_DOCS / "figures"
PATH_DOCS.mkdir(parents=True, exist_ok=True)
PATH_FIGS.mkdir(parents=True, exist_ok=True)

print("PATH_PERFILADO =", PATH_PERFILADO)
print("PATH_FIGS      =", PATH_FIGS)


## 2. Carga, **dtypes A0** y gates de calidad

`read_csv` solo infiere tipos genéricos (`object` / `int64`).  
El perfilado A0 ya definió el contrato; aquí se **reaplican** los casts al cargar (sin re-perfilar texto):

| Columna | dtype canónico A0 |
|---------|-------------------|
| `id_fragmento`, `titulo_origen`, `texto_*`, `hash_texto` | `string` |
| `categoria_l1` | `category` (orden L1 del proyecto) |
| `fecha_extraccion` | `datetime64` |
| `pagina`, `longitud_caracteres`, `conteo_tokens` | `int32` |
| `densidad_lexica` | `float64` |


In [ ]:
def load_perfilado(path: Path) -> pd.DataFrame:
    """Carga el CSV entregado y restaura dtypes del profiling A0."""
    raw = pd.read_csv(path)
    missing = [c for c in SCHEMA if c not in raw.columns]
    if missing:
        raise ValueError(f"Faltan columnas del contrato perfilado: {missing}")

    df = raw.copy()
    # --- strings (evita object opaco) ---
    for col in (
        "id_fragmento",
        "titulo_origen",
        "texto_crudo",
        "hash_texto",
        "texto_limpio",
    ):
        df[col] = df[col].astype("string")

    # --- taxonomía L1 ordenada (category) ---
    df["categoria_l1"] = pd.Categorical(
        df["categoria_l1"].astype(str), categories=L1_ORDER, ordered=True
    )

    # --- temporales y enteros compactos (mismo criterio que profile_and_cast) ---
    df["fecha_extraccion"] = pd.to_datetime(df["fecha_extraccion"], utc=False)
    df["pagina"] = df["pagina"].astype("int32")
    df["longitud_caracteres"] = df["longitud_caracteres"].astype("int32")
    df["conteo_tokens"] = df["conteo_tokens"].astype("int32")
    df["densidad_lexica"] = df["densidad_lexica"].astype("float64")
    return df


df = load_perfilado(PATH_PERFILADO)
print("shape =", df.shape)
print("dtypes (post-cast A0):")
print(df.dtypes)

# --- Gates (hard-fail = datos no aptos para EDA de producto) ---
nulls = df[SCHEMA].isna().sum()
assert (nulls == 0).all(), f"Nulos en schema:\n{nulls[nulls > 0]}"

assert int(df["hash_texto"].duplicated().sum()) == 0, "hash_texto duplicados"
assert int(df["id_fragmento"].duplicated().sum()) == 0, "id_fragmento duplicados"
assert list(df["categoria_l1"].cat.categories) == L1_ORDER, "orden L1 distinto al contrato"
assert set(df["categoria_l1"].astype(str).unique()) == set(L1_ORDER), "L1 distintas al contrato"

# dtypes esperados (regresión de contrato)
assert str(df["categoria_l1"].dtype) == "category"
assert pd.api.types.is_datetime64_any_dtype(df["fecha_extraccion"])
assert df["pagina"].dtype == "int32"
assert df["longitud_caracteres"].dtype == "int32"
assert df["conteo_tokens"].dtype == "int32"
assert df["densidad_lexica"].dtype == "float64"
assert all(df[c].dtype == "string" for c in (
    "id_fragmento", "titulo_origen", "texto_crudo", "hash_texto", "texto_limpio"
))

if len(df) != EXPECTED_ROWS:
    warnings.warn(
        f"n={len(df)} (snapshot de entrega esperaba {EXPECTED_ROWS})",
        UserWarning,
        stacklevel=1,
    )
    print(f"[WARN] n={len(df)} (snapshot {EXPECTED_ROWS})")
else:
    print(f"[OK] n={len(df)} (== EXPECTED_ROWS)")

print("n_origenes (PDF) =", int(df["titulo_origen"].nunique()))
display(
    df[["longitud_caracteres", "conteo_tokens", "densidad_lexica"]]
    .describe()
    .round(2)
)
df.head(2)


## 3. Balance de la variable objetivo (`categoria_l1`)

**Univariado categórico.** Ratio max/min ≈ 1 implica balance plano (diseño V5 opción A ~150/L1).


In [ ]:
vc = df["categoria_l1"].astype(str).value_counts().reindex(L1_ORDER)
bal = pd.DataFrame({"n": vc, "pct": (vc / vc.sum() * 100).round(2)})
ratio_l1 = float(bal["n"].max() / bal["n"].min())
display(bal)
print("ratio max/min L1 =", round(ratio_l1, 4))

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(
    data=bal.reset_index(names="categoria_l1"),
    x="categoria_l1",
    y="n",
    order=L1_ORDER,
    color="#4C72B0",
    ax=ax,
)
ax.axhline(bal["n"].mean(), color="gray", ls="--", lw=1, label="media")
ax.set_title("Balance de fragmentos por categoria_l1")
ax.set_xlabel("")
ax.set_ylabel("n fragmentos")
ax.tick_params(axis="x", rotation=25)
ax.legend(loc="upper right")
fig.tight_layout()
display(fig)  # garantiza PNG embebido en Jupyter / VS Code
fig.savefig(PATH_FIGS / "01_balance_l1.png", bbox_inches="tight")
plt.close(fig)
print("[OK] figura →", PATH_FIGS / "01_balance_l1.png")


## 4. Estructura por documento (`titulo_origen`)

**Pregunta:** ¿una L1 está dominada por pocos PDF?

Si `top1_share` es alto, un split aleatorio por *fila* filtra estilo del mismo PDF a train y test  
(supuesto i.i.d. roto). El EDA **solo recomienda** `GroupShuffleSplit` para la fase de modelado; **no** lo ejecuta.


In [ ]:
rows = []
for l1, g in df.groupby("categoria_l1", observed=False):
    vc_o = g["titulo_origen"].astype(str).value_counts()
    rows.append({
        "categoria_l1": str(l1),
        "n_origenes": int(vc_o.size),
        "top1_origen": str(vc_o.index[0]),
        "top1_n": int(vc_o.iloc[0]),
        "top1_share_pct": round(100.0 * vc_o.iloc[0] / len(g), 1),
        "top3_share_pct": round(100.0 * vc_o.iloc[:3].sum() / len(g), 1),
    })
conc = pd.DataFrame(rows).sort_values("top1_share_pct", ascending=False)
recommend_group_split = bool((conc["top1_share_pct"] > 40).any())
n_origenes = int(df["titulo_origen"].nunique())
frags_per_doc = df.groupby("titulo_origen").size()

display(conc)
print("recommend_group_split =", recommend_group_split, "(umbral top1_share > 40%)")
print("frags/PDF: median =", float(frags_per_doc.median()), "max =", int(frags_per_doc.max()))

fig, ax = plt.subplots(figsize=(9, 4.8))
plot_df = conc.sort_values("top1_share_pct", ascending=True)
sns.barplot(
    data=plot_df,
    y="categoria_l1",
    x="top1_share_pct",
    color="#55A868",
    ax=ax,
    orient="h",
)
ax.axvline(40, color="crimson", ls="--", lw=1, label="umbral 40%")
ax.set_title("% del origen dominante dentro de cada L1")
ax.set_xlabel("top1_share_pct")
ax.legend(loc="lower right")
fig.tight_layout()
display(fig)
fig.savefig(PATH_FIGS / "03_concentracion_origen.png", bbox_inches="tight")
plt.close(fig)
print("[OK] figura →", PATH_FIGS / "03_concentracion_origen.png")


## 5. Longitud, tokens A0 y densidad (univariado + por L1)

- `longitud_caracteres` / `conteo_tokens` / `densidad_lexica` vienen del perfilado A0.  
- Boxplots por L1: detectar clases con pasajes sistemáticamente más cortos o densos.


In [ ]:
num_cols = ["longitud_caracteres", "conteo_tokens", "densidad_lexica"]
display(df[num_cols].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(2))

# Pares de correlaciones (Pearson) — multivariado numérico simple
corr = df[num_cols].corr(numeric_only=True).round(3)
print("correlación Pearson:")
display(corr)

fig, axes = plt.subplots(1, 3, figsize=(14, 5.2))
for ax, col in zip(axes, num_cols):
    sns.boxplot(
        data=df,
        x=col,
        y="categoria_l1",
        order=L1_ORDER,
        orient="h",
        ax=ax,
        color="#4C72B0",
        fliersize=2,
    )
    ax.set_title(col)
    ax.set_ylabel("")
fig.suptitle("Distribución de métricas de texto por L1", y=1.02)
fig.tight_layout()
display(fig)
fig.savefig(PATH_FIGS / "02_longitud.png", bbox_inches="tight")
plt.close(fig)

# Segunda figura dedicada: histograma de conteo_tokens (global)
fig2, ax2 = plt.subplots(figsize=(9, 4))
sns.histplot(df["conteo_tokens"], bins=40, kde=True, color="#8172B3", ax=ax2)
ax2.axvline(df["conteo_tokens"].median(), color="black", ls="--", label="mediana")
ax2.set_title("Distribución global de conteo_tokens (A0)")
ax2.legend()
fig2.tight_layout()
display(fig2)
fig2.savefig(PATH_FIGS / "04_conteo_tokens_por_l1.png", bbox_inches="tight")
plt.close(fig2)
print("[OK] figuras longitud/tokens guardadas")


## 6. Vocabulario del `texto_limpio` (sin sklearn)

Conteos con el mismo `TOKEN_PATTERN` del proyecto A0:

- tamaño de vocabulario, % hapax → hipótesis de `min_df` en un futuro TF-IDF  
- top términos globales  
- términos con **lift** simple por L1 (frecuencia relativa en clase / global)


In [ ]:
def tokens_of(text: object) -> list[str]:
    if not isinstance(text, str) or not text:
        return []
    return [m.group(0).lower() for m in TOKEN_RE.finditer(text)]


tok_lists = df["texto_limpio"].map(tokens_of)
term_freq: Counter[str] = Counter()
doc_freq: Counter[str] = Counter()
for toks in tok_lists:
    term_freq.update(toks)
    doc_freq.update(set(toks))

V = len(term_freq)
hapax = sum(1 for _, c in term_freq.items() if c == 1)
hapax_pct = 100.0 * hapax / V if V else 0.0
min_df_table = {md: sum(1 for _, d in doc_freq.items() if d >= md) for md in (1, 2, 3, 5)}

print(f"|vocab|={V}  hapax={hapax} ({hapax_pct:.1f}%)")
print("min_df table (doc frequency ≥ k):", min_df_table)
print("top 15 términos:", term_freq.most_common(15))

# Lift por L1 (top 8 con df global ≥ 5 para estabilidad)
min_df_lift = 5
global_n = len(df)
top_by_l1: dict[str, list[str]] = {}
for l1 in L1_ORDER:
    mask = df["categoria_l1"].astype(str) == l1
    sub = tok_lists[mask]
    tf_l1: Counter[str] = Counter()
    for toks in sub:
        tf_l1.update(set(toks))  # doc-freq dentro de L1
    n_l1 = int(mask.sum())
    scores: list[tuple[float, str]] = []
    for term, df_l1 in tf_l1.items():
        df_g = doc_freq[term]
        if df_g < min_df_lift:
            continue
        p_l1 = df_l1 / n_l1
        p_g = df_g / global_n
        lift = p_l1 / p_g if p_g > 0 else 0.0
        scores.append((lift, term))
    scores.sort(reverse=True)
    top_by_l1[l1] = [t for _, t in scores[:8]]
    print(f"{l1}: {', '.join(top_by_l1[l1])}")

# Gráfico: hapax vs vocab retenido al subir min_df
fig, ax = plt.subplots(figsize=(7, 4))
xs = list(min_df_table.keys())
ys = [min_df_table[k] for k in xs]
sns.lineplot(x=xs, y=ys, marker="o", ax=ax, color="#C44E52")
ax.set_xlabel("min_df (documentos)")
ax.set_ylabel("|vocabulario| retenido")
ax.set_title("Tamaño de vocabulario vs umbral min_df")
ax.set_xticks(xs)
fig.tight_layout()
display(fig)
fig.savefig(PATH_FIGS / "05_vocab_min_df.png", bbox_inches="tight")
plt.close(fig)
print("[OK] figura vocab →", PATH_FIGS / "05_vocab_min_df.png")


## 7. Flags residuales de calidad de texto (descriptivo)

Heurísticas ligeras sobre `texto_crudo` (TOC, listas de comandos, pies de figura).  
No eliminan filas: solo **inventario** para handoff.


In [ ]:
raw = df["texto_crudo"].astype(str)

def flag_toc(s: str) -> bool:
    return len(re.findall(r"\. \. \.|\.{4,}", s)) >= 3

def flag_cmd(s: str) -> bool:
    return len(re.findall(r" - : ", s)) >= 3 and len(s) < 800

def flag_figura(s: str) -> bool:
    return s.lstrip().startswith("Figura ") and len(s) < 400

flags = pd.DataFrame({
    "toc": raw.map(flag_toc),
    "cmd_list": raw.map(flag_cmd),
    "figura": raw.map(flag_figura),
})
flag_counts = flags.sum().astype(int)
display(flag_counts.to_frame("n_fragmentos"))

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.barplot(x=flag_counts.index, y=flag_counts.values, ax=ax, color="#CCB974")
ax.set_title("Flags residuales (conteo de fragmentos)")
ax.set_ylabel("n")
fig.tight_layout()
display(fig)
fig.savefig(PATH_FIGS / "06_flags_residuales.png", bbox_inches="tight")
plt.close(fig)


## 8. Inventario de columnas (meta para fases posteriores)

Solo **descripción** del contrato de datos. No construye índice ni entrena modelos.


In [ ]:
meta_cols = [
    "id_fragmento",
    "hash_texto",
    "titulo_origen",
    "categoria_l1",
    "texto_limpio",
    "conteo_tokens",
]
present = {c: c in df.columns for c in meta_cols}
display(pd.Series(present, name="presente").to_frame())
assert all(present.values()), "Faltan columnas meta en el perfilado"
print("[OK] contrato meta completo para handoff posterior (TF-IDF / recuperación)")
